In [1]:
import gymnasium as gym
import ale_py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
import cv2
import matplotlib.pyplot as plt
from collections import deque

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.backends.cudnn.benchmark = True
try:
    torch.set_float32_matmul_precision("high")
except:
    pass

Using device: cuda


In [2]:
env = gym.make("ALE/Freeway-v5", render_mode=None)
n_actions = env.action_space.n
print("Actions:", n_actions)

Actions: 3


In [3]:
def extract_base_state(obs):
    gray = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)

    # Approximate chicken search region
    search = gray[20:195, 35:125]
    cy_rel, cx_rel = np.unravel_index(np.argmax(search), search.shape)
    chicken_y = cy_rel + 20
    chicken_x = cx_rel + 35

    # Fixed lane rows for traffic summary
    lane_rows = [32, 48, 64, 80, 96, 112, 128, 144, 160, 176]
    traffic_features = []

    for y in lane_rows:
        row = gray[y:y+1, :]
        small = cv2.resize(row, (12, 1), interpolation=cv2.INTER_AREA).flatten() / 255.0
        traffic_features.extend(small.tolist())

    base = np.array(
        [chicken_x / 160.0, chicken_y / 210.0] + traffic_features,
        dtype=np.float32
    )
    return base

def extract_state(obs, prev_base_state=None):
    base = extract_base_state(obs)

    if prev_base_state is None:
        dy = 0.0
        traffic_delta = np.zeros_like(base[2:], dtype=np.float32)
    else:
        dy = base[1] - prev_base_state[1]
        traffic_delta = base[2:] - prev_base_state[2:]

    full_state = np.concatenate([
        base,
        np.array([dy], dtype=np.float32),
        traffic_delta.astype(np.float32)
    ]).astype(np.float32)

    return full_state, base

input_dim = 243  # 2 + 120 + 1 + 120
print("Input dim:", input_dim)

Input dim: 243


In [4]:
def extract_base_state(obs):
    gray = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)

    # Approximate chicken search region
    search = gray[20:195, 35:125]
    cy_rel, cx_rel = np.unravel_index(np.argmax(search), search.shape)
    chicken_y = cy_rel + 20
    chicken_x = cx_rel + 35

    # Fixed lane rows for traffic summary
    lane_rows = [32, 48, 64, 80, 96, 112, 128, 144, 160, 176]
    traffic_features = []

    for y in lane_rows:
        row = gray[y:y+1, :]
        small = cv2.resize(row, (12, 1), interpolation=cv2.INTER_AREA).flatten() / 255.0
        traffic_features.extend(small.tolist())

    base = np.array(
        [chicken_x / 160.0, chicken_y / 210.0] + traffic_features,
        dtype=np.float32
    )
    return base

def extract_state(obs, prev_base_state=None):
    base = extract_base_state(obs)

    if prev_base_state is None:
        dy = 0.0
        traffic_delta = np.zeros_like(base[2:], dtype=np.float32)
    else:
        dy = base[1] - prev_base_state[1]
        traffic_delta = base[2:] - prev_base_state[2:]

    full_state = np.concatenate([
        base,
        np.array([dy], dtype=np.float32),
        traffic_delta.astype(np.float32)
    ]).astype(np.float32)

    return full_state, base

input_dim = 243  # 2 + 120 + 1 + 120
print("Input dim:", input_dim)

Input dim: 243


In [5]:
def extract_base_state(obs):
    gray = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)

    # Approximate chicken search region
    search = gray[20:195, 35:125]
    cy_rel, cx_rel = np.unravel_index(np.argmax(search), search.shape)
    chicken_y = cy_rel + 20
    chicken_x = cx_rel + 35

    # Fixed lane rows for traffic summary
    lane_rows = [32, 48, 64, 80, 96, 112, 128, 144, 160, 176]
    traffic_features = []

    for y in lane_rows:
        row = gray[y:y+1, :]
        small = cv2.resize(row, (12, 1), interpolation=cv2.INTER_AREA).flatten() / 255.0
        traffic_features.extend(small.tolist())

    base = np.array(
        [chicken_x / 160.0, chicken_y / 210.0] + traffic_features,
        dtype=np.float32
    )
    return base

def extract_state(obs, prev_base_state=None):
    base = extract_base_state(obs)

    if prev_base_state is None:
        dy = 0.0
        traffic_delta = np.zeros_like(base[2:], dtype=np.float32)
    else:
        dy = base[1] - prev_base_state[1]
        traffic_delta = base[2:] - prev_base_state[2:]

    full_state = np.concatenate([
        base,
        np.array([dy], dtype=np.float32),
        traffic_delta.astype(np.float32)
    ]).astype(np.float32)

    return full_state, base

input_dim = 243  # 2 + 120 + 1 + 120
print("Input dim:", input_dim)

Input dim: 243


In [6]:
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        return self.net(x)

In [7]:
class ReplayBuffer:
    def __init__(self, capacity=100000):
        self.buffer = deque(maxlen=capacity)

    def add(self, s, a, r, s2, d):
        self.buffer.append((s, a, r, s2, d))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, s2, d = zip(*batch)
        return (
            torch.tensor(np.array(s), dtype=torch.float32, device=device),
            torch.tensor(a, dtype=torch.long, device=device),
            torch.tensor(r, dtype=torch.float32, device=device),
            torch.tensor(np.array(s2), dtype=torch.float32, device=device),
            torch.tensor(d, dtype=torch.float32, device=device),
        )

    def __len__(self):
        return len(self.buffer)

In [8]:
def custom_reward(prev_state, next_state, env_reward, done):
    prev_y = prev_state[1] * 210.0
    next_y = next_state[1] * 210.0

    reward = 0.0

    # Small reward for upward movement
    if next_y < prev_y:
        reward += 0.15 * (prev_y - next_y)

    # Penalty for moving downward
    elif next_y > prev_y:
        reward -= 0.10 * (next_y - prev_y)

    # BIG reward for real crossing from game
    reward += 100.0 * float(env_reward)

    # Tiny time penalty so it does not waste steps
    reward -= 0.02

    if done:
        reward -= 0.5

    return reward

In [9]:
def choose_action(state, epsilon, model):
    if random.random() < epsilon:
        return random.choices(
            population=[0, 1, 2],
            weights=[0.15, 0.70, 0.15],
            k=1
        )[0]
    else:
        with torch.no_grad():
            s = torch.tensor(np.array(state), dtype=torch.float32, device=device).unsqueeze(0)
            return model(s).argmax(dim=1).item()

In [10]:
model = DQN(input_dim, n_actions).to(device)
target_model = DQN(input_dim, n_actions).to(device)
target_model.load_state_dict(model.state_dict())

optimizer = optim.Adam(model.parameters(), lr=5e-4)
buffer = ReplayBuffer(100000)

gamma = 0.99
batch_size = 128
updates_per_step = 4
epsilon = 1.0
epsilon_decay = 0.992
epsilon_min = 0.08
episodes = 500
target_update_every = 10
max_steps_per_episode = 1000

episode_rewards = []
episode_crossings = []
episode_steps = []
episode_avg_steps_per_crossing = []

best_avg_cross = -1.0
best_single_cross = -1

In [11]:
for ep in range(episodes):
    obs, _ = env.reset()
    prev_base_state = None
    state, prev_base_state = extract_state(obs, prev_base_state)

    total_reward = 0.0
    crossings = 0

    for step in range(max_steps_per_episode):
        action = choose_action(state, epsilon, model)

        next_obs, env_reward, done, truncated, info = env.step(action)
        next_state, next_base_state = extract_state(next_obs, prev_base_state)

        reward = custom_reward(state, next_state, env_reward, done or truncated)

        if env_reward > 0:
            crossings += 1

        buffer.add(state, action, reward, next_state, float(done or truncated))

        state = next_state
        prev_base_state = next_base_state
        total_reward += reward

        if len(buffer) >= batch_size:
            for _ in range(updates_per_step):
                s, a, r, s2, d = buffer.sample(batch_size)

                q = model(s).gather(1, a.unsqueeze(1)).squeeze(1)

                # Double DQN target
                with torch.no_grad():
                    next_actions = model(s2).argmax(dim=1, keepdim=True)
                    q_next = target_model(s2).gather(1, next_actions).squeeze(1)
                    target = r + gamma * q_next * (1.0 - d)

                loss = nn.MSELoss()(q, target)

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()

        if done or truncated:
            break

    # log metrics
    episode_rewards.append(total_reward)
    episode_crossings.append(crossings)
    episode_steps.append(step + 1)

    if crossings > 0:
        episode_avg_steps_per_crossing.append((step + 1) / crossings)
    else:
        episode_avg_steps_per_crossing.append(np.nan)

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if ep % target_update_every == 0:
        target_model.load_state_dict(model.state_dict())

    recent_avg_cross = np.mean(episode_crossings[-25:]) if len(episode_crossings) >= 25 else np.mean(episode_crossings)

    # Save best checkpoint by average crossings
    if recent_avg_cross > best_avg_cross:
        best_avg_cross = recent_avg_cross
        torch.save(model.state_dict(), "freeway_best.pth")
        print(f"New best avg-cross checkpoint saved at episode {ep} | avg_cross_25={best_avg_cross:.2f}")

    # Save best single-episode crossing model too
    if crossings > best_single_cross:
        best_single_cross = crossings
        torch.save(model.state_dict(), "freeway_best_single_cross.pth")
        print(f"New best single-cross checkpoint saved at episode {ep} | crossings={best_single_cross}")

    if ep % 25 == 0:
        print(f"Ep {ep:3d} | reward {total_reward:8.2f} | crossings {crossings} | avg_cross_25 {recent_avg_cross:.2f} | eps {epsilon:.3f}")

New best avg-cross checkpoint saved at episode 0 | avg_cross_25=6.00
New best single-cross checkpoint saved at episode 0 | crossings=6
Ep   0 | reward   613.10 | crossings 6 | avg_cross_25 6.00 | eps 0.992
New best avg-cross checkpoint saved at episode 1 | avg_cross_25=6.50
New best single-cross checkpoint saved at episode 1 | crossings=7
New best avg-cross checkpoint saved at episode 4 | avg_cross_25=6.80
New best single-cross checkpoint saved at episode 4 | crossings=8
New best avg-cross checkpoint saved at episode 5 | avg_cross_25=6.83
New best avg-cross checkpoint saved at episode 7 | avg_cross_25=6.88
New best avg-cross checkpoint saved at episode 8 | avg_cross_25=6.89
New best avg-cross checkpoint saved at episode 9 | avg_cross_25=6.90
New best avg-cross checkpoint saved at episode 10 | avg_cross_25=6.91
New best avg-cross checkpoint saved at episode 11 | avg_cross_25=7.00
Ep  25 | reward   514.10 | crossings 5 | avg_cross_25 6.76 | eps 0.812
New best single-cross checkpoint save

KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), "freeway_final.pth")
print("Saved freeway_final.pth")

In [ ]:
eval_model = DQN(input_dim, n_actions).to(device)
eval_model.load_state_dict(torch.load("freeway_best_single_cross.pth", map_location=device))
eval_model.eval()
print("Loaded best evaluation model.")

In [ ]:
eval_env = gym.make("ALE/Freeway-v5", render_mode="rgb_array")

obs, _ = eval_env.reset()
prev_base_state = None
state, prev_base_state = extract_state(obs, prev_base_state)

frames = []
crossings = 0
steps = 0

for step in range(2000):
    steps += 1

    with torch.no_grad():
        s = torch.tensor(np.array(state), dtype=torch.float32, device=device).unsqueeze(0)
        action = eval_model(s).argmax(dim=1).item()

    obs, env_reward, done, truncated, info = eval_env.step(action)
    next_state, next_base_state = extract_state(obs, prev_base_state)

    frames.append(obs)

    if env_reward > 0:
        crossings += 1
        print(f"Crossing #{crossings} at step {steps}")

    state = next_state
    prev_base_state = next_base_state

    if done or truncated:
        break

eval_env.close()

print("\n===== FINAL RESULTS =====")
print("Crossings:", crossings)
print("Steps taken:", steps)
print("Crossings per 100 steps:", crossings / max(steps, 1) * 100)
print("Average steps per crossing:", steps / crossings if crossings > 0 else "N/A")

In [ ]:
cross_so_far = 0
scored_frames = []

eval_env = gym.make("ALE/Freeway-v5", render_mode="rgb_array")

obs, _ = eval_env.reset()
prev_base_state = None
state, prev_base_state = extract_state(obs, prev_base_state)

for step in range(2000):
    with torch.no_grad():
        s = torch.tensor(np.array(state), dtype=torch.float32, device=device).unsqueeze(0)
        action = eval_model(s).argmax(dim=1).item()

    obs, env_reward, done, truncated, info = eval_env.step(action)
    next_state, next_base_state = extract_state(obs, prev_base_state)

    if env_reward > 0:
        cross_so_far += 1

    frame = obs.copy()
    cv2.putText(frame, f"Crossings: {cross_so_far}", (8, 24),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(frame, f"Step: {step}", (8, 48),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    scored_frames.append(frame)

    state = next_state
    prev_base_state = next_base_state

    if done or truncated:
        break

eval_env.close()

if len(scored_frames) > 0:
    h, w, _ = scored_frames[0].shape
    out = cv2.VideoWriter(
        "freeway_scored.mp4",
        cv2.VideoWriter_fourcc(*"mp4v"),
        20,
        (w, h)
    )

    for f in scored_frames:
        out.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))

    out.release()
    print("Saved freeway_scored.mp4")
else:
    print("No frames were collected.")

In [ ]:
def moving_avg(x, window=25):
    x = np.array(x, dtype=float)
    if len(x) == 0:
        return np.array([])
    if len(x) < window:
        return np.array([np.mean(x)])
    return np.convolve(x, np.ones(window) / window, mode="valid")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(episode_rewards, alpha=0.35, label="Episode Reward")

ma_rewards = moving_avg(episode_rewards, 25)
x_ma = range(24, len(episode_rewards)) if len(episode_rewards) >= 25 else [len(episode_rewards)-1]
plt.plot(x_ma, ma_rewards, linewidth=2.5, label="25-Episode Moving Average")

plt.title("Training Reward Over Time")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(episode_crossings, alpha=0.4, label="Episode Crossings")

ma_cross = moving_avg(episode_crossings, 25)
x_ma = range(24, len(episode_crossings)) if len(episode_crossings) >= 25 else [len(episode_crossings)-1]
plt.plot(x_ma, ma_cross, linewidth=2.5, label="25-Episode Moving Average")

plt.title("Successful Crossings Over Time")
plt.xlabel("Episode")
plt.ylabel("Crossings")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
clean_steps_per_cross = np.array(episode_avg_steps_per_crossing, dtype=float)

plt.figure(figsize=(10, 5))
plt.plot(clean_steps_per_cross, alpha=0.4, label="Episode Avg Steps per Crossing")

valid_idx = ~np.isnan(clean_steps_per_cross)
valid_vals = clean_steps_per_cross[valid_idx]
valid_positions = np.where(valid_idx)[0]

if len(valid_vals) > 0:
    ma_steps = moving_avg(valid_vals, 25)
    x_ma = valid_positions[24:] if len(valid_vals) >= 25 else [valid_positions[-1]]
    plt.plot(x_ma, ma_steps, linewidth=2.5, label="25-Episode Moving Average")

plt.title("Efficiency: Average Steps Per Crossing")
plt.xlabel("Episode")
plt.ylabel("Steps Per Crossing (Lower is Better)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
rolling_best = np.maximum.accumulate(episode_crossings)

plt.figure(figsize=(10, 5))
plt.plot(rolling_best, linewidth=2.5)
plt.title("Best Crossing Count Achieved So Far")
plt.xlabel("Episode")
plt.ylabel("Best Crossings")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
total_steps_collected = int(np.sum(episode_steps))
total_episodes = len(episode_steps)

print("Total episodes:", total_episodes)
print("Total interaction steps collected:", total_steps_collected)
print("Best episode crossings:", np.max(episode_crossings))
print("Average crossings per episode:", np.mean(episode_crossings))
print("Best reward:", np.max(episode_rewards))